# 读取DataPattern下的文件

注意：所有JPG图像都会被认为是训练集。

In [1]:
import os
import pandas as pd

from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

data_pattern = get_param_in_cwd('data_pattern')
patches = []
for r, ds, fs in os.walk(data_pattern):
    patches.extend([os.path.join(r, p) for p in fs if p.endswith('.jpg')])

patches = pd.DataFrame(patches, columns=['fpath'])
patches['fname'] = patches['fpath'].map(lambda x: os.path.basename(x))

def map2id(x):
    x = os.path.basename(x)
    *fname_items, w, h, ltx, lty, rbx, rby = x.split('_')
    return '_'.join(fname_items)
patches['ID'] = patches['fpath'].map(map2id)
patches

,fpath,fname,ID
0,E:\\qymnew\\JPG\1\1_000_000_000_000_000_003.jpg,1_000_000_000_000_000_003.jpg,1
1,E:\\qymnew\\JPG\1\1_000_000_000_000_000_004.jpg,1_000_000_000_000_000_004.jpg,1
2,E:\\qymnew\\JPG\1\1_000_000_000_000_000_006.jpg,1_000_000_000_000_000_006.jpg,1
3,E:\\qymnew\\JPG\1\1_000_000_000_000_000_007.jpg,1_000_000_000_000_000_007.jpg,1
4,E:\\qymnew\\JPG\1\1_000_000_000_000_000_008.jpg,1_000_000_000_000_000_008.jpg,1
...,...,...,...
8611,E:\\qymnew\\JPG\98\98_000_000_000_000_000_070.jpg,98_000_000_000_000_000_070.jpg,98
8612,E:\\qymnew\\JPG\98\98_000_000_000_000_000_071.jpg,98_000_000_000_000_000_071.jpg,98
8613,E:\\qymnew\\JPG\98\98_000_000_000_000_000_073.jpg,98_000_000_000_000_000_073.jpg,98
8614,E:\\qymnew\\JPG\98\98_000_000_000_000_000_074.jpg,98_000_000_000_000_000_074.jpg,98


# 获取划分的数据集

In [2]:
label_f = pd.read_csv(get_param_in_cwd('label_file'))
label_f['ID'] = label_f['ID'].map(lambda x: str(x).replace('.nii.gz', ''))
patches = pd.merge(patches, label_f, on='ID', how='inner')
patches

,fpath,fname,ID,label,group
0,E:\\qymnew\\JPG\1\1_000_000_000_000_000_003.jpg,1_000_000_000_000_000_003.jpg,1,0,train
1,E:\\qymnew\\JPG\1\1_000_000_000_000_000_004.jpg,1_000_000_000_000_000_004.jpg,1,0,train
2,E:\\qymnew\\JPG\1\1_000_000_000_000_000_006.jpg,1_000_000_000_000_000_006.jpg,1,0,train
3,E:\\qymnew\\JPG\1\1_000_000_000_000_000_007.jpg,1_000_000_000_000_000_007.jpg,1,0,train
4,E:\\qymnew\\JPG\1\1_000_000_000_000_000_008.jpg,1_000_000_000_000_000_008.jpg,1,0,train
...,...,...,...,...,...
8611,E:\\qymnew\\JPG\98\98_000_000_000_000_000_070.jpg,98_000_000_000_000_000_070.jpg,98,0,train
8612,E:\\qymnew\\JPG\98\98_000_000_000_000_000_071.jpg,98_000_000_000_000_000_071.jpg,98,0,train
8613,E:\\qymnew\\JPG\98\98_000_000_000_000_000_073.jpg,98_000_000_000_000_000_073.jpg,98,0,train
8614,E:\\qymnew\\JPG\98\98_000_000_000_000_000_074.jpg,98_000_000_000_000_000_074.jpg,98,0,train


# 保存List模式数据

In [3]:
import numpy as np

train_patches = patches[patches['group'] == 'train']
train_patches[['fpath', 'label']].to_csv(os.path.join('split_info', f'train.txt'), index=False, header=False, sep='\t')
test_patches = patches[patches['group'] != 'train']
test_patches[['fpath', 'label']].to_csv(os.path.join('split_info', f'val.txt'), index=False, header=False, sep='\t')